<a href="https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session4/solutions/cci_session4_biomedical_ner_finetuning_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 CCI Session 4 — Fine-Tuning a Biomedical NER Model

**Cancer Care Informatics (CCI) — Prompt Engineering & Clinical AI Development**  
**King Hussein Cancer Center (KHCC) · AI & Data Intelligence Department (AIDI)**  
**Instructor: Iyad Sultan, MD — CAIO & CHIIO**

---

### What you will learn

| Section | Topic |
|---------|-------|
| **Part 1** | Load & explore the `d4data/biomedical-ner-all` baseline model |
| **Part 2** | Prepare an NER dataset (NCBI Disease or BC5CDR) for fine-tuning |
| **Part 3** | Fine-tune with HuggingFace `Trainer` — entity-level P / R / F1 |
| **Part 4** | Inference on pathology-style tumor registry text |
| **Part 5** | Written comparison — open-source NER vs. OpenAI extraction (Session 3) |
| **Stretch** | Multi-model benchmark · SQLite JSON export · Alternate clinical task |

> ⚠️ **Runtime**: Go to **Runtime → Change runtime type → T4 GPU** before running.

---
## 0 · Environment Setup

In [ ]:
%%capture
!pip install -q seqeval evaluate peft
!pip install -q sentencepiece protobuf

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import torch
import numpy as np
import json, time, os
from pprint import pprint

# Confirm GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
else:
    print("⚠️  No GPU — fine-tuning will be very slow. Switch to T4 runtime.")

---
## Part 1 · Baseline Model — `d4data/biomedical-ner-all`

This model is a fine-tuned BERT model trained on **22 biomedical NER datasets** covering diseases, chemicals, genes, species, cell lines, and more.  
Let's load it and see what it can already do before any additional fine-tuning.

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

MODEL_NAME = "d4data/biomedical-ner-all"

tokenizer_baseline = AutoTokenizer.from_pretrained(MODEL_NAME)
model_baseline = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

ner_pipeline = pipeline(
    "ner",
    model=model_baseline,
    tokenizer=tokenizer_baseline,
    aggregation_strategy="simple",
    device=0 if device == "cuda" else -1,
)

print(f"\n📋 Label map ({len(model_baseline.config.id2label)} labels):")
for idx, label in sorted(model_baseline.config.id2label.items()):
    print(f"  {idx}: {label}")

### 1.1 · Baseline inference on pathology-style text

These are synthetic sentences modeled after the kind of text you'd find in a KHCC tumor registry or pathology report.

In [ ]:
# Synthetic pathology / tumor registry examples
PATHOLOGY_SAMPLES = [
    """Invasive ductal carcinoma, grade 2, ER positive, PR positive, HER2 negative.
    Tumor size 2.3 cm. Sentinel lymph node biopsy: 1 of 3 nodes positive for metastatic carcinoma.""",

    """Right nephrectomy specimen: Clear cell renal cell carcinoma, Fuhrman grade 3,
    measuring 5.8 x 4.2 x 3.9 cm, confined to the kidney. Renal vein margin negative.""",

    """Bone marrow biopsy: Hypercellular marrow (90%) with sheets of blasts consistent
    with acute myeloid leukemia (AML), FAB M4. Flow cytometry positive for CD13, CD33, CD117.""",

    """Excisional biopsy of left cervical lymph node: Classical Hodgkin lymphoma,
    nodular sclerosis subtype. Reed-Sternberg cells positive for CD15 and CD30.""",

    """Wilms tumor (nephroblastoma), favorable histology, Stage III.
    Tumor extends beyond the kidney capsule with positive surgical margins.
    Patient started on vincristine, dactinomycin, and doxorubicin (DD4A regimen).""",
]

print("=" * 80)
print("BASELINE MODEL PREDICTIONS ON PATHOLOGY TEXT")
print("=" * 80)

for i, text in enumerate(PATHOLOGY_SAMPLES):
    print(f"\n--- Sample {i+1} ---")
    print(f"Text: {text[:120].strip()}...\n")
    results = ner_pipeline(text)
    for ent in results:
        print(f"  [{ent['entity_group']:20s}] {ent['word']:30s}  (score: {ent['score']:.3f})")
    if not results:
        print("  (no entities detected)")

### 1.2 · Baseline observations

**✏️ Student Exercise:** Run the cell above and answer:
1. Which entity types does the baseline model detect well?
2. What does it miss that a tumor registrar would care about (stage, grade, margins, receptor status)?
3. Does it confuse disease names with chemical/drug names?

---
## Part 2 · Prepare a Fine-Tuning Dataset

We'll use the **NCBI Disease** corpus — 793 PubMed abstracts annotated with disease mentions.  
This is one of the standard benchmarks for biomedical NER.

We load directly from HuggingFace's **auto-converted Parquet files** (the `refs/convert/parquet`  
branch). This avoids the legacy loading script that recent `datasets` versions no longer support.

> **Swap dataset:** To use BC5CDR (diseases + chemicals) instead, replace the Parquet URLs  
> with the equivalent from `tner/bc5cdr` and update `LABEL_NAMES` accordingly.

In [ ]:
from datasets import load_dataset, DatasetDict

# ==========================================================================
# NCBI Disease corpus loaded from HuggingFace's auto-converted Parquet files.
# The original 'ncbi_disease' repo uses a legacy loading script that is no
# longer supported by recent versions of the `datasets` library. Loading
# from the refs/convert/parquet branch bypasses the script entirely.
# ==========================================================================

PARQUET_BASE = (
    "https://huggingface.co/datasets/ncbi/ncbi_disease/resolve/"
    "refs%2Fconvert%2Fparquet/ncbi_disease/"
)

raw_dataset = load_dataset(
    "parquet",
    data_files={
        "train":      PARQUET_BASE + "train/0000.parquet",
        "validation": PARQUET_BASE + "validation/0000.parquet",
        "test":       PARQUET_BASE + "test/0000.parquet",
    },
)

DATASET_DISPLAY = "NCBI Disease (Parquet mirror)"

print(f"\n📊 Dataset: {DATASET_DISPLAY}")
print(raw_dataset)
print(f"\nColumn names: {raw_dataset['train'].column_names}")
print(f"\nSample entry:")
pprint(raw_dataset["train"][0])

In [ ]:
# Inspect the NER tag set
#
# The Parquet-converted dataset stores ner_tags as plain integer lists
# (no ClassLabel metadata), so we define the label names explicitly.
# NCBI Disease uses BIO tagging: 0 = O, 1 = B-Disease, 2 = I-Disease

LABEL_NAMES = ["O", "B-Disease", "I-Disease"]

NUM_LABELS = len(LABEL_NAMES)
label2id = {lbl: i for i, lbl in enumerate(LABEL_NAMES)}
id2label = {i: lbl for i, lbl in enumerate(LABEL_NAMES)}

print(f"Label names: {LABEL_NAMES}")
print(f"Number of labels: {NUM_LABELS}")

# Verify tag distribution
from collections import Counter
tag_counts = Counter()
for ex in raw_dataset["train"]:
    tag_counts.update(ex["ner_tags"])
print(f"\nTag distribution (train):")
for tag_id, count in sorted(tag_counts.items()):
    print(f"  {tag_id} ({id2label[tag_id]:10s}): {count:,}")

### 2.1 · Tokenize & align labels

BERT uses WordPiece tokenization — a single word may split into multiple sub-tokens.  
We need to propagate the NER label to the first sub-token and set **-100** on the rest  
(so the loss function ignores them).

In [ ]:
# Use a fresh tokenizer for fine-tuning (same architecture, clean config)
FINETUNE_BASE = "dmis-lab/biobert-base-cased-v1.2"  # BioBERT — strong biomedical baseline

tokenizer = AutoTokenizer.from_pretrained(FINETUNE_BASE)

def tokenize_and_align_labels(examples):
    """Tokenize tokens (already split) and align NER tags to sub-tokens."""
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=256,
        padding="max_length",
    )

    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                # Special tokens ([CLS], [SEP], [PAD])
                label_ids.append(-100)
            elif word_id != prev_word_id:
                # First sub-token of a word → use original label
                label_ids.append(labels[word_id])
            else:
                # Continuation sub-token → -100 (ignored in loss)
                label_ids.append(-100)
            prev_word_id = word_id
        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized


tokenized_datasets = raw_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_dataset["train"].column_names,
)

print("✅ Tokenization complete")
print(f"   Train: {len(tokenized_datasets['train']):,} examples")
print(f"   Val:   {len(tokenized_datasets['validation']):,} examples")
print(f"   Test:  {len(tokenized_datasets['test']):,} examples")

In [ ]:
# Sanity check — decode one example and show alignment
example = tokenized_datasets["train"][0]
tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
labels = example["labels"]

print("Token-label alignment (first 40 tokens):")
print(f"{'Token':20s} {'Label ID':>10s}  {'Label Name':15s}")
print("-" * 50)
for tok, lbl in zip(tokens[:40], labels[:40]):
    lbl_name = id2label[lbl] if lbl != -100 else "(ignored)"
    print(f"{tok:20s} {lbl:>10d}  {lbl_name:15s}")

---
## Part 3 · Fine-Tune with HuggingFace Trainer

We fine-tune **BioBERT** on the disease NER dataset using the HuggingFace `Trainer` API.  
Evaluation uses `seqeval` — the standard library for **entity-level** (not token-level) metrics.

In [ ]:
import evaluate
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

# Load seqeval metric
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    """Entity-level precision, recall, F1 using seqeval."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Remove padding (-100) and convert IDs → label strings
    true_labels = []
    true_preds  = []

    for pred_seq, label_seq in zip(preds, labels):
        t_labels = []
        t_preds  = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                t_labels.append(id2label[l])
                t_preds.append(id2label[p])
        true_labels.append(t_labels)
        true_preds.append(t_preds)

    results = seqeval_metric.compute(
        predictions=true_preds,
        references=true_labels,
        mode="strict",
        scheme="IOB2",
    )

    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

In [ ]:
# Initialize a fresh BioBERT model with the correct number of output labels
model = AutoModelForTokenClassification.from_pretrained(
    FINETUNE_BASE,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

# Data collator — handles dynamic padding for NER
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print(f"\n🏗️  Model: {FINETUNE_BASE}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Trainable:  {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"   Output labels: {NUM_LABELS} → {LABEL_NAMES}")

In [ ]:
OUTPUT_DIR = "./biomedical-ner-finetuned"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    # Training
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=88, # Replaced warmup_ratio with calculated warmup_steps
    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    # Logging
    logging_steps=50,
    report_to="none",
    # Performance
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Trainer configured — ready to train")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch:  {training_args.per_device_train_batch_size}")
print(f"   LR:     {training_args.learning_rate}")
print(f"   FP16:   {training_args.fp16}")

### 📏 Baseline evaluation — BEFORE any training

Before we train, let's evaluate the randomly-initialized classification head on the test set.  
This gives us a proper **before vs. after** comparison using the exact same metric and label schema.

In [ ]:
# Evaluate the model BEFORE training (random classifier head on top of BioBERT)
print("Evaluating model BEFORE training (epoch 0)...")
pre_train_results = trainer.evaluate(tokenized_datasets["test"])

print("\n" + "=" * 60)
print("📊  BEFORE TRAINING — TEST SET METRICS (seqeval, strict)")
print("=" * 60)
print(f"   Precision:  {pre_train_results['eval_precision']:.4f}")
print(f"   Recall:     {pre_train_results['eval_recall']:.4f}")
print(f"   F1:         {pre_train_results['eval_f1']:.4f}")
print(f"   Accuracy:   {pre_train_results['eval_accuracy']:.4f}")
print("=" * 60)
print("\n💡 These numbers reflect a randomly initialized classification head.")
print("   The BioBERT encoder knows biomedical language, but the 3-class")
print("   NER head hasn't learned what a disease entity looks like yet.")

In [ ]:
# 🚀 Train!
print("Training started...")
start_time = time.time()

train_result = trainer.train()

elapsed = time.time() - start_time
print(f"\n✅ Training complete in {elapsed / 60:.1f} minutes")
print(f"   Final train loss: {train_result.training_loss:.4f}")

In [ ]:
# Evaluate on the held-out TEST set — AFTER training
post_train_results = trainer.evaluate(tokenized_datasets["test"])

print("\n" + "=" * 70)
print("📊  BEFORE vs. AFTER TRAINING — TEST SET (seqeval, strict)")
print("=" * 70)
print(f"{'Metric':<15s} {'BEFORE':>12s} {'AFTER':>12s} {'Δ':>12s}")
print("-" * 52)
for metric_key, label in [("eval_precision", "Precision"),
                           ("eval_recall", "Recall"),
                           ("eval_f1", "F1"),
                           ("eval_accuracy", "Accuracy")]:
    before = pre_train_results[metric_key]
    after  = post_train_results[metric_key]
    delta  = after - before
    print(f"{label:<15s} {before:>12.4f} {after:>12.4f} {delta:>+12.4f}")
print("=" * 70)

# Store for use in the comparison table later
test_results = post_train_results

### 3.1 · Per-entity-type breakdown

For datasets with multiple entity types (e.g., BC5CDR with Disease + Chemical), we get a per-type breakdown.

In [ ]:
# Run a detailed per-type evaluation
eval_pred = trainer.predict(tokenized_datasets["test"])
logits = eval_pred.predictions
labels = eval_pred.label_ids
preds = np.argmax(logits, axis=-1)

true_labels = []
true_preds  = []
for pred_seq, label_seq in zip(preds, labels):
    t_l, t_p = [], []
    for p, l in zip(pred_seq, label_seq):
        if l != -100:
            t_l.append(id2label[l])
            t_p.append(id2label[p])
    true_labels.append(t_l)
    true_preds.append(t_p)

# Full seqeval report
from seqeval.metrics import classification_report
print(classification_report(true_labels, true_preds, digits=4))

---
## Part 4 · Inference on Pathology / Tumor Registry Text

Now let's run our fine-tuned model on the same pathology samples we tested with the baseline.

In [ ]:
# Build a pipeline from the fine-tuned model
finetuned_pipeline = pipeline(
    "ner",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if device == "cuda" else -1,
)

print("=" * 80)
print("FINE-TUNED MODEL PREDICTIONS ON PATHOLOGY TEXT")
print("=" * 80)

finetuned_results = []

for i, text in enumerate(PATHOLOGY_SAMPLES):
    print(f"\n--- Sample {i+1} ---")
    print(f"Text: {text[:120].strip()}...\n")

    t0 = time.time()
    results = finetuned_pipeline(text)
    latency = (time.time() - t0) * 1000

    finetuned_results.append({"text": text, "entities": results, "latency_ms": latency})

    for ent in results:
        print(f"  [{ent['entity_group']:20s}] {ent['word']:35s}  (score: {ent['score']:.3f})")
    print(f"  ⏱ Latency: {latency:.1f} ms")

In [ ]:
# Side-by-side comparison: baseline vs. fine-tuned
print("\n" + "=" * 90)
print("SIDE-BY-SIDE: BASELINE d4data/biomedical-ner-all  vs.  FINE-TUNED BioBERT")
print("=" * 90)

for i, text in enumerate(PATHOLOGY_SAMPLES):
    print(f"\n{'─' * 90}")
    print(f"Sample {i+1}: {text[:100].strip()}...")
    print()

    baseline_ents = ner_pipeline(text)
    finetuned_ents = finetuned_pipeline(text)

    print(f"  {'BASELINE':40s} │ {'FINE-TUNED':40s}")
    print(f"  {'─'*40} │ {'─'*40}")

    max_len = max(len(baseline_ents), len(finetuned_ents))
    for j in range(max_len):
        left  = f"{baseline_ents[j]['entity_group']}: {baseline_ents[j]['word']}" if j < len(baseline_ents) else ""
        right = f"{finetuned_ents[j]['entity_group']}: {finetuned_ents[j]['word']}" if j < len(finetuned_ents) else ""
        print(f"  {left:40s} │ {right:40s}")

### 4.1 · Custom text input

Try your own pathology text here.

In [ ]:
# ✏️ Replace this with your own text
custom_text = """Embryonal rhabdomyosarcoma of the right orbit, Stage I, Group IIa.
Margins positive at the medial extent. No lymph node involvement.
Patient will receive vincristine and actinomycin D per COG protocol ARST0331."""

print("\n🔬 Custom text NER results:\n")
for ent in finetuned_pipeline(custom_text):
    print(f"  [{ent['entity_group']:20s}] {ent['word']:35s}  (score: {ent['score']:.3f})")

---
## Part 5 · Comparison: Open-Source NER vs. OpenAI Extraction

In **Session 3**, we used OpenAI's API (GPT-4 / GPT-4o-mini) with structured output to extract
clinical entities from pathology text using prompt engineering. Let's compare the two approaches systematically.

### 5.1 · Simulated OpenAI extraction (for reference)

In [ ]:
# This cell simulates what we did in Session 3 with OpenAI.
# If you have an API key, uncomment the live version below.

# --- Simulated OpenAI structured extraction result (Session 3 pattern) ---
openai_simulated_result = {
    "sample": PATHOLOGY_SAMPLES[0],
    "extracted": {
        "diagnosis": "Invasive ductal carcinoma",
        "grade": "Grade 2",
        "er_status": "Positive",
        "pr_status": "Positive",
        "her2_status": "Negative",
        "tumor_size_cm": 2.3,
        "lymph_nodes_positive": 1,
        "lymph_nodes_total": 3,
        "procedure": "Sentinel lymph node biopsy",
    },
    "latency_ms": 1800,    # Typical GPT-4o-mini latency
    "cost_per_call_usd": 0.003,  # ~750 input + 200 output tokens at GPT-4o-mini rates
}

print("📋 Simulated OpenAI extraction (Session 3 pattern):")
pprint(openai_simulated_result["extracted"])
print(f"\n⏱ Latency: {openai_simulated_result['latency_ms']} ms")
print(f"💰 Cost: ${openai_simulated_result['cost_per_call_usd']:.4f} per note")

In [ ]:
# --- OPTIONAL: Live OpenAI call (uncomment if you have an API key) ---

# import openai
# openai.api_key = "sk-..."  # Or use google.colab.userdata
#
# EXTRACTION_PROMPT = """Extract the following fields from this pathology report.
# Return ONLY valid JSON with these keys:
# diagnosis, grade, er_status, pr_status, her2_status, tumor_size_cm,
# lymph_nodes_positive, lymph_nodes_total, procedure
#
# Pathology text:
# {text}"""
#
# t0 = time.time()
# response = openai.ChatCompletion.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": EXTRACTION_PROMPT.format(text=PATHOLOGY_SAMPLES[0])}],
#     temperature=0,
# )
# live_latency = (time.time() - t0) * 1000
# print(f"Live OpenAI result ({live_latency:.0f} ms):")
# print(response.choices[0].message.content)

### 5.2 · Structured comparison table

In [ ]:
# Measure fine-tuned model's average latency
latencies = []
for text in PATHOLOGY_SAMPLES:
    t0 = time.time()
    _ = finetuned_pipeline(text)
    latencies.append((time.time() - t0) * 1000)

avg_latency_local = np.mean(latencies)

print(f"Fine-tuned model avg latency: {avg_latency_local:.1f} ms per note")
print(f"OpenAI GPT-4o-mini avg latency: ~1500-2500 ms per note")

In [ ]:
# Build comparison dataframe
import pandas as pd

comparison = pd.DataFrame([
    {
        "Dimension": "Model",
        "Fine-Tuned BioBERT (NER)": "BioBERT + NCBI Disease",
        "OpenAI GPT-4o-mini (Extraction)": "GPT-4o-mini + prompt",
    },
    {
        "Dimension": "Task type",
        "Fine-Tuned BioBERT (NER)": "Token classification (BIO tags)",
        "OpenAI GPT-4o-mini (Extraction)": "Generative structured extraction",
    },
    {
        "Dimension": "Entity types",
        "Fine-Tuned BioBERT (NER)": "Disease only (or Disease+Chemical)",
        "OpenAI GPT-4o-mini (Extraction)": "Any schema you define in the prompt",
    },
    {
        "Dimension": "Latency (per note)",
        "Fine-Tuned BioBERT (NER)": f"{avg_latency_local:.0f} ms (GPU)",
        "OpenAI GPT-4o-mini (Extraction)": "~1500–2500 ms (API)",
    },
    {
        "Dimension": "Cost (10K notes)",
        "Fine-Tuned BioBERT (NER)": "$0 (self-hosted) or ~$5 GPU",
        "OpenAI GPT-4o-mini (Extraction)": "~$30 (GPT-4o-mini)",
    },
    {
        "Dimension": "Privacy / PHI",
        "Fine-Tuned BioBERT (NER)": "✅ Runs on-prem / no data leaves",
        "OpenAI GPT-4o-mini (Extraction)": "⚠️ Data sent to OpenAI API",
    },
    {
        "Dimension": "Schema flexibility",
        "Fine-Tuned BioBERT (NER)": "❌ Fixed label set (retrain to change)",
        "OpenAI GPT-4o-mini (Extraction)": "✅ Change prompt → change schema",
    },
    {
        "Dimension": "Accuracy on disease NER",
        "Fine-Tuned BioBERT (NER)": f"F1: {test_results['eval_f1']:.3f} (strict eval)",
        "OpenAI GPT-4o-mini (Extraction)": "~0.85–0.92 (varies by prompt)",
    },
    {
        "Dimension": "Handles relations",
        "Fine-Tuned BioBERT (NER)": "❌ NER only — no relations",
        "OpenAI GPT-4o-mini (Extraction)": "✅ Can extract relations, reasoning",
    },
    {
        "Dimension": "Hallucination risk",
        "Fine-Tuned BioBERT (NER)": "None — only classifies existing tokens",
        "OpenAI GPT-4o-mini (Extraction)": "Non-zero — can invent entities",
    },
    {
        "Dimension": "Best for",
        "Fine-Tuned BioBERT (NER)": "High-volume, privacy-sensitive tagging",
        "OpenAI GPT-4o-mini (Extraction)": "Complex extraction, prototyping",
    },
])

# Display
comparison = comparison.set_index("Dimension")
comparison.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

### 5.3 · Written analysis

**Cost:** For a hospital processing 10,000+ clinical notes per month, the fine-tuned BioBERT model running on a single GPU is essentially free after the initial training cost. OpenAI API calls at GPT-4o-mini pricing would run around $30/month at that volume — manageable, but it scales linearly. Using GPT-4o for higher accuracy increases the cost 10–15x.

**Privacy:** This is the decisive factor for KHCC and most hospitals. Patient pathology reports contain protected health information (PHI). A fine-tuned model runs entirely within the hospital's Azure infrastructure — data never leaves the firewall. OpenAI's API requires sending PHI to external servers, which triggers IRB, HIPAA, and Jordanian data protection concerns regardless of OpenAI's data use policies.

**Latency:** The fine-tuned model processes a pathology note in 10–50 ms on a T4 GPU. OpenAI API calls take 1.5–3 seconds including network round-trip. For real-time applications (e.g., a pathologist reviewing a case and needing inline NER annotations), only the local model is fast enough.

**Accuracy & Flexibility:** This is where LLM extraction has a clear advantage. GPT-4o can extract *arbitrary* structured fields (diagnosis, stage, grade, margins, receptor status, treatment plan) from a single prompt — the schema is defined in natural language and can be changed instantly. The NER model only identifies *entity spans* within a fixed label set. To extract new entity types, you need new training data and a retraining cycle.

**KHCC Recommendation:** Use **both** in a hybrid pipeline. The fine-tuned NER model runs first as a fast, private, high-precision disease tagger on all incoming notes. For complex cases that need structured extraction (staging, grading, treatment regimen), route the note to an Azure OpenAI instance within the hospital's own Azure tenant — keeping PHI inside the firewall while leveraging LLM flexibility.

---
## 🏋️ Stretch Options

Choose one (or more) of the following challenges.

### Stretch A · Multi-Model Benchmark

Compare **3 models** on the same test set:
1. `d4data/biomedical-ner-all` (baseline, no fine-tuning)
2. Your fine-tuned BioBERT
3. A different base model (e.g., `allenai/scibert_scivocab_cased`)

In [ ]:
# Stretch A — Multi-model benchmark
# Uncomment and run to compare multiple models

# MODELS_TO_BENCHMARK = {
#     "d4data/biomedical-ner-all (baseline)": "d4data/biomedical-ner-all",
#     "BioBERT fine-tuned (ours)": OUTPUT_DIR,
#     # Add more models here:
#     # "SciBERT": "allenai/scibert_scivocab_cased",
# }
#
# benchmark_results = []
#
# for model_name, model_path in MODELS_TO_BENCHMARK.items():
#     print(f"\nEvaluating: {model_name}")
#     try:
#         pipe = pipeline(
#             "ner", model=model_path, tokenizer=model_path,
#             aggregation_strategy="simple",
#             device=0 if device == "cuda" else -1,
#         )
#         # Time on sample texts
#         times = []
#         all_ents = []
#         for text in PATHOLOGY_SAMPLES:
#             t0 = time.time()
#             ents = pipe(text)
#             times.append((time.time() - t0) * 1000)
#             all_ents.append(len(ents))
#         benchmark_results.append({
#             "Model": model_name,
#             "Avg entities/sample": np.mean(all_ents),
#             "Avg latency (ms)": np.mean(times),
#         })
#     except Exception as e:
#         print(f"  ⚠️ Skipped: {e}")
#
# if benchmark_results:
#     pd.DataFrame(benchmark_results).set_index("Model")

### Stretch B · SQLite JSON Export

Save all NER predictions to a SQLite database with the original text, extracted entities as JSON, model name, and timestamp — simulating a production pipeline output.

In [ ]:
# Stretch B — SQLite export
import sqlite3
from datetime import datetime

DB_PATH = "ner_predictions.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS ner_extractions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        source_text TEXT NOT NULL,
        model_name TEXT NOT NULL,
        entities_json TEXT NOT NULL,
        entity_count INTEGER,
        latency_ms REAL,
        created_at TEXT DEFAULT CURRENT_TIMESTAMP
    )
""")

# Insert predictions
for result in finetuned_results:
    entities_serializable = [
        {
            "entity_group": e["entity_group"],
            "word": e["word"],
            "score": round(float(e["score"]), 4),
            "start": e["start"],
            "end": e["end"],
        }
        for e in result["entities"]
    ]
    cursor.execute(
        "INSERT INTO ner_extractions (source_text, model_name, entities_json, entity_count, latency_ms) VALUES (?, ?, ?, ?, ?)",
        (
            result["text"],
            f"biobert-finetuned-ncbi-disease",
            json.dumps(entities_serializable),
            len(entities_serializable),
            result["latency_ms"],
        ),
    )

conn.commit()

# Verify
rows = cursor.execute("SELECT id, model_name, entity_count, latency_ms FROM ner_extractions").fetchall()
print(f"\n✅ Saved {len(rows)} predictions to {DB_PATH}")
print(f"\n{'ID':>4s}  {'Model':40s}  {'Entities':>8s}  {'Latency':>10s}")
print("-" * 70)
for row in rows:
    print(f"{row[0]:>4d}  {row[1]:40s}  {row[2]:>8d}  {row[3]:>8.1f} ms")

# Show one full JSON record
sample_json = cursor.execute("SELECT entities_json FROM ner_extractions LIMIT 1").fetchone()[0]
print(f"\nSample entities JSON:")
pprint(json.loads(sample_json))

conn.close()

### Stretch C · Alternate clinical task — Medication NER

Apply the same pipeline to a different clinical task: detecting medication mentions.  
This uses `d4data/biomedical-ner-all` which already has medication labels.

In [ ]:
# Stretch C — Medication detection using the baseline model's broader label set

medication_texts = [
    "Patient started on cisplatin 75mg/m2 day 1 and etoposide 100mg/m2 days 1-3, "
    "every 21 days for 4 cycles. Ondansetron 8mg IV pre-chemo for nausea prophylaxis.",

    "Maintenance therapy with temozolomide 150mg/m2 for 5 days, 28-day cycle. "
    "Concurrent dexamethasone 4mg BID for cerebral edema. PPI omeprazole 20mg daily.",

    "Immunotherapy: Pembrolizumab 200mg IV every 3 weeks. Thyroid function monitoring "
    "with TSH every 6 weeks. Hold if grade 3+ immune-related adverse events.",
]

print("MEDICATION NER — Baseline d4data model")
print("=" * 80)

for i, text in enumerate(medication_texts):
    print(f"\n--- Medication Sample {i+1} ---")
    results = ner_pipeline(text)
    for ent in results:
        print(f"  [{ent['entity_group']:20s}] {ent['word']:30s}  ({ent['score']:.3f})")

---
## 📦 Save & Download the Fine-Tuned Model

In [ ]:
# Save the final model
SAVE_DIR = "./biomedical-ner-final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"\n✅ Model saved to {SAVE_DIR}")
print(f"   Files:")
for f in sorted(os.listdir(SAVE_DIR)):
    size_mb = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f"   {f:40s}  {size_mb:.1f} MB")

In [ ]:
# Optional: zip and download
# !zip -r biomedical-ner-final.zip ./biomedical-ner-final
# from google.colab import files
# files.download("biomedical-ner-final.zip")

---
## Summary & Key Takeaways

| What we did | Why it matters |
|-------------|----------------|
| Loaded a pre-trained biomedical NER model | Understand what's available off-the-shelf |
| Tokenized & aligned BIO labels for sub-word tokens | The critical step most tutorials skip |
| Fine-tuned BioBERT with HuggingFace Trainer | Reproducible training with entity-level eval |
| Evaluated with seqeval strict P/R/F1 | Gold-standard NER evaluation (not token accuracy) |
| Compared to OpenAI extraction (Session 3) | Understand the cost/privacy/flexibility tradeoffs |

### When to use which approach at KHCC

| Scenario | Recommended approach |
|----------|---------------------|
| High-volume disease tagging (all pathology notes) | Fine-tuned NER on Azure Databricks |
| Complex structured extraction (staging, margins) | Azure OpenAI (GPT-4.1-mini in KHCC tenant) |
| Prototyping a new extraction schema | OpenAI / Claude with prompt iteration |
| PHI-sensitive research cohort extraction | Fine-tuned model or Azure OpenAI (on-prem) |
| Real-time clinical decision support | Fine-tuned model (latency <50ms) |

---

**Next session:** LangChain / LangGraph — building agentic pipelines that combine NER, RAG, and LLM reasoning.